# 🧪 Notebook 03 — Jordan–Wigner Mapping

> **Module:** Quantum Computing for Computational Chemistry\
> **Level:** Intermediate\
> **Estimated time:** 30 min\
> **Prerequisites:** Notebook 01, second quantization, Hartree–Fock, and matrix diagonalization

---

## 📌 Overview

This notebook maps a fermionic Hamiltonian to qubit Pauli operators using the Jordan–Wigner transformation.

## 🎯 Learning objectives

By the end of this notebook, you should be able to:

- Transform atomic-orbital integrals into molecular-orbital integrals.
- Implement Jordan–Wigner mappings for fermionic operators.
- Assemble and diagonalize a qubit Hamiltonian.

## 🔑 Key concepts

`Jordan–Wigner transformation` · `spin orbitals` · `parity strings` · `molecular integrals` · `Pauli Hamiltonian` · `FCI`

## 🧩 Structure

1. H₂ Hartree–Fock calculation
2. Molecular-orbital integrals
3. Jordan–Wigner transformation
4. Hamiltonian construction
5. Spectrum and particle-number sectors

## 📚 Resources

- PySCF, OpenFermion, and Qiskit documentation
- Previous notebook: `openfermion.ipynb`

### ⏱️ Expected outcome

You will construct a four-qubit H₂ Pauli Hamiltonian and compare its spectrum with the molecular FCI benchmark.

# Jordan-Wigner mapping

This notebook builds a molecular Hamiltonian and maps it from fermionic creation and annihilation operators to qubit Pauli operators usig Jordan Wigner transformation. The workflow is:

1. Solve the electronic structure problem with PySCF.
2. Transform one- and two-electron integrals to the molecular-orbital basis.
3. Encode the fermionic operators with the Jordan-Wigner transformation.
4. Assemble the qubit Hamiltonian and inspect its matrix spectrum.

The example uses H2 in the STO-3G basis, which has two spatial orbitals and therefore four spin orbitals, requiring four qubits.

In [1]:
import numpy as np
import qiskit
import openfermion
import pyscf
from openfermion import FermionOperator, QubitOperator


## H2 molecule in a minimal basis

In [2]:
# H2 molecule
from pyscf import gto, scf

# Build molecule
mol = gto.M(
    atom=[["H", (0.0, 0.0, 0.0)],
          ["H", (0.0, 0.0, 0.74)]],
    basis="sto-3g",
    unit="Angstrom",
)

# Hartree-Fock calculation
mf = scf.RHF(mol)
mf.kernel()

converged SCF energy = -1.11675930739643


np.float64(-1.1167593073964255)

The nuclei are separated by 0.74 Angstrom, and the `sto-3g` basis supplies one contracted spatial orbital per hydrogen atom. Restricted Hartree-Fock (RHF) assumes paired electrons in the same spatial orbitals and provides the molecular-orbital coefficients used below. The printed SCF energy is the converged Hartree-Fock total electronic energy for this geometry and basis.

## One- and two-electron integrals

The core Hamiltonian contains the one-electron kinetic-energy and electron-nuclear-attraction terms. The `int2e` tensor contains two-electron repulsion integrals in the atomic-orbital (AO) basis. These quantities are the numerical coefficients from which the second-quantized molecular Hamiltonian is assembled.

In [3]:
import numpy as np

# Core Hamiltonian (kinetic + nuclear attraction)
h1 = mf.get_hcore()
print(h1)

[[-1.12095946 -0.95937577]
 [-0.95937577 -1.12095946]]


In [4]:
eri = mol.intor("int2e")
print(eri.shape)

(2, 2, 2, 2)


## Transforming to molecular orbitals

The qubit Hamiltonian is built from molecular-orbital (MO) integrals rather than AO integrals. If $C$ is the RHF coefficient matrix, the one-electron transformation is $h^{MO}=C^T h^{AO}C$. `ao2mo.kernel` performs the analogous transformation for the four-index electron-repulsion tensor, and the transpose puts its indices into the convention used by the operator expansion below.

In [5]:
from pyscf import ao2mo

C = mf.mo_coeff

# One-electron integrals in MO basis
h1 = mf.get_hcore() # Core Hamiltonian (kinetic + nuclear attraction)
h1_mo = C.T @ h1 @ C

# Two-electron integrals in MO basis (Electron Repulsion Integrals)
eri_mo = ao2mo.kernel(mol, C)
eri_mo = ao2mo.restore(1, eri_mo, C.shape[1])
# h[p,q,r,s] = (ps|qr)
eri_mo = eri_mo.transpose(0, 2, 3, 1)

# 1 electron integrals
print(h1_mo)
# 2 electron integrals
print(eri_mo)

[[-1.25330979e+00 -2.16420228e-16]
 [-4.69718139e-16 -4.75068849e-01]]
[[[[ 6.74755927e-01  5.55111512e-17]
   [-8.32667268e-17  1.81210462e-01]]

  [[-8.32667268e-17  1.81210462e-01]
   [ 6.63711401e-01  3.60822483e-16]]]


 [[[ 5.55111512e-17  6.63711401e-01]
   [ 1.81210462e-01  5.27355937e-16]]

  [[ 1.81210462e-01  5.27355937e-16]
   [ 3.60822483e-16  6.97651504e-01]]]]


The nearly zero off-diagonal entries in the one-electron MO matrix are a consequence of the canonical RHF orbitals. The two spatial orbitals are the bonding and antibonding orbitals. The transformed ERI tensor supplies the interaction coefficients for pairs of spin orbitals.

In [6]:
e_nuc = mf.energy_nuc()
e_nuc

np.float64(0.7151043390810812)

`e_nuc` is the classical nuclear-nuclear repulsion energy. It contributes a scalar multiple of the identity to the electronic Hamiltonian, so it shifts every eigenvalue by the same amount without changing eigenvectors.

## Jordan-Wigner mapping

\begin{aligned}
a_i^\dagger =
\frac{1}{2}(X_i-iY_i)
\prod_{j<i}Z_j
\end{aligned}

\begin{aligned}
a_i =
\frac{1}{2}(X_i+iY_i)
\prod_{j<i}Z_j.
\end{aligned}


The factors $X_i\mp iY_i$ change the occupation of qubit $i$. The parity string $\prod_{j<i}Z_j$ records the parity of all lower-indexed modes; it is essential because fermionic operators must anticommute on different modes. Under this convention, one qubit represents one spin orbital, and computational-basis occupation states represent fermionic Fock states.

In [7]:
def creation(i):

    parity = 1
    for j in range(i):
        parity *= QubitOperator(f'Z{j}')

    return 0.5 * (QubitOperator(f'X{i}') - 1j * QubitOperator(f'Y{i}')) * parity

def annihilation(i):

    parity = 1
    for j in range(i):
        parity *= QubitOperator(f'Z{j}')

    return 0.5 * (QubitOperator(f'X{i}') + 1j * QubitOperator(f'Y{i}')) * parity


`creation(i)` and `annihilation(i)` construct symbolic `QubitOperator` expressions for a single spin orbital. The loop builds the Jordan-Wigner parity string one $Z$ factor at a time. OpenFermion combines products of Pauli operators and collects equal Pauli words, leaving a form that can be evaluated or converted to a matrix.

## Building the second-quantized Hamiltonian

For two spatial orbitals, the four spin orbitals are ordered as $0\alpha, 0\beta, 1\alpha, 1\beta$. The electronic Hamiltonian has the form

$$H = E_{nuc}+\sum_{pq\sigma}h_{pq}a^\dagger_{p\sigma}a_{q\sigma}+\frac{1}{2}\sum_{pqrs\sigma\tau}h_{pqrs}a^\dagger_{p\sigma}a^\dagger_{q\tau}a_{r\tau}a_{s\sigma}.$$

The loops substitute the Jordan-Wigner expressions for these operators. The separate $\alpha\alpha$, $\alpha\beta$, $\beta\alpha$, and $\beta\beta$ terms account for all spin combinations in the two-electron contribution.

In [8]:
H_qubit = QubitOperator(()) * e_nuc

for i in range(2):
    for j in range(2):
        # α
        H_qubit += h1_mo[i, j] * creation(2*i) * annihilation(2*j)
        # β
        H_qubit += h1_mo[i, j] * creation(2*i+1) * annihilation(2*j+1)

# Two-electron part
for p in range(2):
    for q in range(2):
        for r in range(2):
            for s in range(2):

                value = 0.5 * eri_mo[p, q, r, s]
                # αα
                H_qubit += value * creation(2*p) * creation(2*q) * annihilation(2*r) * annihilation(2*s)

                # αβ
                H_qubit += value * creation(2*p) * creation(2*q+1) * annihilation(2*r+1) * annihilation(2*s)

                # βα
                H_qubit += value * creation(2*p+1) * creation(2*q) * annihilation(2*r) * annihilation(2*s+1)

                # ββ
                H_qubit += value * creation(2*p+1) * creation(2*q+1) * annihilation(2*r+1) * annihilation(2*s+1)

H_qubit

(-0.09706626816763159+0j) [] +
(-0.04530261550379926+0j) [X0 X1 Y2 Y3] +
(0.04530261550379926+0j) [X0 Y1 Y2 X3] +
(0.04530261550379926+0j) [Y0 X1 X2 Y3] +
(-0.04530261550379926+0j) [Y0 Y1 X2 X3] +
(0.1714128264477691+0j) [Z0] +
(0.16868898170361207+0j) [Z0 Z1] +
(0.12062523483390412+0j) [Z0 Z2] +
(0.16592785033770335+0j) [Z0 Z3] +
(0.1714128264477691+0j) [Z1] +
(0.16592785033770335+0j) [Z1 Z2] +
(0.12062523483390412+0j) [Z1 Z3] +
(-0.2234315369081343+0j) [Z2] +
(0.17441287612261575+0j) [Z2 Z3] +
(-0.2234315369081343+0j) [Z3]

The result is a sum of Pauli words such as $Z_0$, $Z_0Z_1$, and four-qubit $X/Y$ strings. The empty Pauli word `[]` is the identity term and includes the nuclear repulsion together with scalar contributions generated by the fermionic products. These terms can be measured individually on a quantum processor or converted to a matrix for classical verification.

In [9]:
from openfermion import get_sparse_operator

H_sparse = get_sparse_operator(H_qubit)
print(H_sparse)

<Compressed Sparse Column sparse matrix of dtype 'complex128'
	with 20 stored elements and shape (16, 16)>
  Coords	Values
  (0, 0)	(0.7151043390810807+0j)
  (1, 1)	(0.24003549030890292+0j)
  (2, 2)	(0.24003549030890303+0j)
  (3, 3)	(0.46261814602718815+0j)
  (12, 3)	(0.18121046201519703+0j)
  (4, 4)	(-0.5382054475648965+0j)
  (5, 5)	(-0.5307733570014579+0j)
  (6, 6)	(-0.349562894986261+0j)
  (9, 6)	(-0.18121046201519703+0j)
  (7, 7)	(0.3555207000676407+0j)
  (8, 8)	(-0.5382054475648965+0j)
  (6, 9)	(-0.18121046201519703+0j)
  (9, 9)	(-0.349562894986261+0j)
  (10, 10)	(-0.5307733570014579+0j)
  (11, 11)	(0.3555207000676407+0j)
  (3, 12)	(0.18121046201519703+0j)
  (12, 12)	(-1.1167593073964253+0j)
  (13, 13)	(-0.44561581548217344+0j)
  (14, 14)	(-0.44561581548217344+0j)
  (15, 15)	(0.9231791809225416+0j)


## Matrix representation

`get_sparse_operator` converts the symbolic Pauli sum into a $2^4\times2^4=16\times16$ sparse matrix because the mapped Hamiltonian acts on four qubits. Rows and columns are indexed by computational-basis occupation states. Sparsity is useful for larger systems because most matrix elements are zero, even though the full matrix dimension grows exponentially with the number of qubits.

In [10]:
np.linalg.eigvals(H_sparse.toarray().real)

array([-0.16835243+0.j, -0.53077336+0.j, -1.13728383+0.j,  0.48314267+0.j,
        0.71510434+0.j,  0.24003549+0.j,  0.24003549+0.j, -0.53820545+0.j,
       -0.53077336+0.j,  0.3555207 +0.j, -0.53820545+0.j, -0.53077336+0.j,
        0.3555207 +0.j, -0.44561582+0.j, -0.44561582+0.j,  0.92317918+0.j])

## Interpreting the eigenvalues

Diagonalizing the matrix gives the energy levels represented by this four-qubit Hamiltonian. The lowest value is the ground-state energy in the chosen basis and geometry. Repeated eigenvalues reflect degeneracies, while the occupied-electron sector must be selected when comparing with a physical H2 calculation: the full qubit matrix also contains states with other particle numbers.

This classical diagonalization is a verification step. A quantum eigensolver would estimate the relevant ground-state energy by preparing a trial state and measuring expectation values of the Pauli terms rather than constructing the full matrix.